In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import wandb

In [2]:
config = {
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "Adam",
    "model": "CNN_BN_FashionMNIST",
}

wandb.init(
    project="fashion-mnist-cnn",   # 项目名，可以自己改
    name="adam-bn-baseline",       # 本次实验名称
    config=config                  # 记录所有超参数
)
# ---------- 1. 准备数据 ----------
# 把图片（0-255像素）转为张量（0-1），并标准化（让数据分布更稳定）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 下载训练集和测试集（第一次运行会下载，耐心等几十秒）
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform,download=True)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=transform,download=True)

# DataLoader：按批次(Batch)喂数据，每批64张，并打乱顺序
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
test_loader = DataLoader(test_set, batch_size=config["batch_size"], shuffle=False)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\35314\_netrc.
wandb: Currently logged in as: 3531427435 (3531427435-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 202tlufg
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in E:\code\PycharmProjects\pythonProject\pytorch\wandb\run-20260912_131707-202tlufg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run adam-bn-baseline
wandb:  View project at https://wandb.ai/3531427435-none/fashion-mnist-cnn
wandb:  View run at https://wandb.ai/3531427435-none/fashion-mnist-cnn/runs/202tlufg


In [3]:
# ---------- 2. 定义模型 ----------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # 卷积层
        # Conv2d（输入通道，输出通道，卷积核大小）
        self.conv1 = nn.Conv2d(1, 16, 5) # 输入1通道，输出16通道，卷积核5x5
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 5) # 输入16通道，输出32通道，卷积核5x5
        self.bn2 = nn.BatchNorm2d(32)
        # 全连接层
        # Linear(输入维度，输出维度)
        self.fc1 = nn.Linear(32 * 4 * 4, 120)
        self.bn3 = nn.BatchNorm1d(120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 输出10类

    def forward(self, x):
        # 卷积1 -> BN -> Relu -> 池化
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, (2,2))

        # 卷积2 -> BN -> Relu -> 池化
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)

        # 展平多维的卷积图成一维的向量
        x = torch.flatten(x, 1)

        # 全连接层 + Relu
        x= F.relu(self.bn3(self.fc1(x)))

        x = F.relu(self.fc2(x))

        # 输出层(10类)
        x = self.fc3(x)
        # print(x.shape)
        return x

net = Net().to(device)
print(net)
wandb.watch(net, log="all", log_freq=100)
# wandb.watch(net, log="gradients", log_freq=100)

使用设备: cuda
Net(
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc1): Linear(in_features=512, out_features=120, bias=True)
  (bn3): BatchNorm1d(120, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [4]:
# ---------- 3. 定义损失函数和优化器 ----------
criterion = nn.CrossEntropyLoss() # 交叉熵损失：专用于分类
# optimizer = optim.SGD(net.parameters(), lr=0.04) # 随机梯度下降，学习率0.04
optimizer = optim.Adam(net.parameters(), lr=config["learning_rate"]) # Adam优化器

In [5]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# ---------- 4. 训练循环（铁打的5步舞曲） ----------
num_epochs = config["epochs"]

for epoch in range(num_epochs):
    net.train()  # <--- 确保训练时 BatchNorm 正常更新
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = net(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(net, test_loader)

    print(
        f"Epoch [{epoch+1}/{config['epochs']}], "
        f"Loss: {avg_loss:.4f}, "
        f"Test Acc: {test_acc:.2f}%"
    )

    # 记录到 W&B
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_loss,
        "test/accuracy": test_acc,
    })

print("Finished Training")

# ---------- 5. 保存模型 Artifact ----------
torch.save(net.state_dict(), "fashion_mnist_cnn.pth")
artifact = wandb.Artifact(
    name="fashion-mnist-cnn",
    type="model"
)
artifact.add_file("fashion_mnist_cnn.pth")
wandb.log_artifact(artifact)
wandb.finish()

Epoch [1/10], Loss: 0.4557, Test Acc: 87.88%
Epoch [2/10], Loss: 0.2875, Test Acc: 89.18%
Epoch [3/10], Loss: 0.2460, Test Acc: 89.62%
Epoch [4/10], Loss: 0.2177, Test Acc: 90.26%
Epoch [5/10], Loss: 0.1956, Test Acc: 90.26%
Epoch [6/10], Loss: 0.1790, Test Acc: 90.68%
Epoch [7/10], Loss: 0.1615, Test Acc: 90.44%
Epoch [8/10], Loss: 0.1482, Test Acc: 90.34%
Epoch [9/10], Loss: 0.1364, Test Acc: 90.40%
Epoch [10/10], Loss: 0.1224, Test Acc: 90.77%
Finished Training


wandb: uploading artifact fashion-mnist-cnn; updating run metadata
wandb: uploading artifact fashion-mnist-cnn
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading config.yaml
wandb: uploading history steps 9-9, summary
wandb: 
wandb: Run history:
wandb:         epoch ▁▂▃▃▄▅▆▆▇█
wandb: test/accuracy ▁▄▅▇▇█▇▇▇█
wandb:    train/loss █▄▄▃▃▂▂▂▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 10
wandb: test/accuracy 90.77
wandb:    train/loss 0.12242
wandb: 
wandb:  View run adam-bn-baseline at: https://wandb.ai/3531427435-none/fashion-mnist-cnn/runs/202tlufg
wandb:  View project at: https://wandb.ai/3531427435-none/fashion-mnist-cnn
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: .\wandb\run-20260912_131707-202tlufg\logs
